In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

import src.utils.utils as ut

In [4]:
wine_df = pd.read_csv(ut.get_project_file_path("src", "data", "transformed", "wines_clean.csv"))
wine_df = wine_df.reset_index(drop=False,)
wine_df = wine_df.rename(columns={"index": "wine_id"})
wine_df.head(3)

,wine_id,wine_link,name,year,winery,rating,rating_qty,price,body,tannins,...,San Carlos,San Juan,San Rafael,Serra Gaúcha,Tulum Valley,Tunuyán,Tupungato,Uco Valley,Vale dos Vinhedos,Vista Flores
0,0,https://www.vivino.com/US/en/luigi-bosca-parai...,Paraiso,2020.0,Luigi Bosca,4.8,582.0,188.33,0.7343,0.5090,...,0,0,0,0,0,0,0,0,0,0
1,1,https://www.vivino.com/US/en/catena-zapata-est...,Estiba Reservada,2015.0,Catena Zapata,4.7,297.0,675.00,0.7417,0.5583,...,0,0,0,0,0,0,0,0,0,0
2,2,https://www.vivino.com/US/en/catena-zapata-est...,Estiba Reservada,2017.0,Catena Zapata,4.7,219.0,580.00,0.7417,0.5583,...,0,0,0,0,0,0,0,0,0,0


In [52]:
users_data = pd.read_pickle(ut.get_project_file_path("src", "data", "synthetic", "simulation_03_n3000.pkl"))
users_data = users_data.dropna()
users_data.head(3)

,user_id,wine_id,user_input,metrics_local,metrics_global,liked,prob_like
0,0,862.0,"{'meal': 'Tortilla de Papa', 'main_pairing': '...",rating_rscld -0.4 price_rscld ...,rating_rscld_gbl -1.0 price_rscl...,0.0,0.732020
2,2,257.0,"{'meal': 'Pastel de Papa', 'main_pairing': 'be...",rating_rscld 0.5 price_rscld ...,rating_rscld_gbl 0.333333 price_rscl...,1.0,0.871839
3,3,1611.0,"{'meal': 'Burrito de Cerdo', 'main_pairing': '...",rating_rscld 0.333333 price_rscld ...,rating_rscld_gbl 0.0 price_rscl...,1.0,0.888836


In [53]:
recommend_df = pd.merge(wine_df, users_data, how="inner", on="wine_id")
recommend_df.head(3)

,wine_id,wine_link,name,year,winery,rating,rating_qty,price,body,tannins,...,Tupungato,Uco Valley,Vale dos Vinhedos,Vista Flores,user_id,user_input,metrics_local,metrics_global,liked,prob_like
0,11,https://www.vivino.com/US/en/catena-zapata-mal...,Malbec Argentino,2020.0,Catena Zapata,4.6,3695.0,66.67,0.677,0.3338,...,0,0,0,0,396,"{'meal': 'Carne con Salsa de Champiñones', 'ma...",rating_rscld 1.0 price_rscld ...,rating_rscld_gbl 2.0 price_rscl...,1.0,0.792001
1,11,https://www.vivino.com/US/en/catena-zapata-mal...,Malbec Argentino,2020.0,Catena Zapata,4.6,3695.0,66.67,0.677,0.3338,...,0,0,0,0,1887,"{'meal': 'Hamburguesa con Queso', 'main_pairin...",rating_rscld 1.25 price_rscld ...,rating_rscld_gbl 2.0 price_rscl...,1.0,0.870516
2,11,https://www.vivino.com/US/en/catena-zapata-mal...,Malbec Argentino,2020.0,Catena Zapata,4.6,3695.0,66.67,0.677,0.3338,...,0,0,0,0,2105,"{'meal': 'Matambre a la Pizza', 'main_pairing'...",rating_rscld 1.0 price_rscld ...,rating_rscld_gbl 2.0 price_rscl...,1.0,0.961758


In [39]:
grapes = pd.read_csv(ut.get_project_file_path("src", "data", "processed", "aux", "grapes.csv"))
grapes_list = list(grapes["grapes"])
region = pd.read_csv(ut.get_project_file_path("src", "data", "processed", "aux", "region.csv"))
region_list = list(region["region"])
pairings = pd.read_csv(ut.get_project_file_path("src", "data", "processed", "aux", "pairings.csv"))
pairings_list = list(pairings["pairings"])

In [55]:
test_df = recommend_df.copy()
test_df["user_main_pairing"] = test_df["user_input"].apply(lambda x: x["main_pairing"])
test_df["has_user_main_pairing"] = test_df.apply(lambda row: 1 if row[row["user_main_pairing"]]==1 else 0, axis=1)

taste_list = ["body", "tannins", "sweetness", "acidity"]
for taste in taste_list:
    test_df[taste + "_min"] = test_df["user_input"].apply(lambda x: x["tastes"][taste][0])
    test_df[taste + "_max"] = test_df["user_input"].apply(lambda x: x["tastes"][taste][1])

test_df["taste_distance_raw"] = test_df["metrics_global"].apply(lambda x: x["taste_distance_raw_gbl"])
test_df





#main_pairing = recommend_df["user_input"]["main_pairing"]
# recommend_df[main_pairing] == 1

# recommend_df["user_input"].loc[0]["tastes"]["body"][0]
# recommend_df["user_input"].loc[0]["tastes"]["body"][1]
# users_data["metrics_global"].loc[0]["taste_distance_raw_gbl"]

,wine_id,wine_link,name,year,winery,rating,rating_qty,price,body,tannins,...,has_user_main_pairing,body_min,body_max,tannins_min,tannins_max,sweetness_min,sweetness_max,acidity_min,acidity_max,taste_distance_raw
0,11,https://www.vivino.com/US/en/catena-zapata-mal...,Malbec Argentino,2020.0,Catena Zapata,4.6,3695.0,66.67,0.6770,0.3338,...,1,0.204380,0.649635,0.000000,0.327146,0.495763,1.000000,0.523209,1.000000,0.192243
1,11,https://www.vivino.com/US/en/catena-zapata-mal...,Malbec Argentino,2020.0,Catena Zapata,4.6,3695.0,66.67,0.6770,0.3338,...,1,0.204380,0.649635,0.327146,0.426016,0.430085,0.495763,0.000000,0.377909,0.054918
2,11,https://www.vivino.com/US/en/catena-zapata-mal...,Malbec Argentino,2020.0,Catena Zapata,4.6,3695.0,66.67,0.6770,0.3338,...,1,0.181898,0.516898,0.307237,0.362927,0.356403,0.440678,0.568857,0.952681,0.444283
3,13,https://www.vivino.com/US/en/el-enemigo-gran-e...,Gran Enemigo Single Vineyard Chacayes Cabernet...,2016.0,El Enemigo,4.6,2716.0,129.99,0.8187,0.5464,...,1,0.955912,1.000000,0.527307,0.656612,0.391714,0.844397,0.707214,0.778835,0.064582
4,22,https://www.vivino.com/US/en/bodegas-bianchi-e...,Enzo Bianchi Gran Corte,2020.0,Bodegas Bianchi,4.5,25.0,70.00,0.8112,0.5782,...,1,0.929781,0.955912,0.672585,0.693301,0.219397,0.307674,0.393725,0.604475,0.316295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2697,1950,https://www.vivino.com/etchart-cafayate-terroi...,Cafayate Terroir de Altura Torrontés,2021.0,Bodegas Etchart,3.9,110.0,16.99,0.4789,0.3701,...,0,0.955912,1.000000,0.672585,0.693301,0.391714,0.844397,0.604475,0.655587,5.849617
2698,1952,https://www.vivino.com/gen-del-alma-jijiji-mal...,JIJIJI Malbec - Pinot Noir,2022.0,Gen del Alma,3.9,105.0,22.66,0.6311,0.4507,...,1,0.659854,0.708613,0.334409,0.434423,0.323211,0.434322,0.000000,0.373923,0.577012
2699,1952,https://www.vivino.com/gen-del-alma-jijiji-mal...,JIJIJI Malbec - Pinot Noir,2022.0,Gen del Alma,3.9,105.0,22.66,0.6311,0.4507,...,1,0.745839,1.000000,0.000000,0.327146,0.495763,1.000000,0.523209,1.000000,0.219450
2700,1953,https://www.vivino.com/terrazas-de-los-andes-h...,High Altitude Vineyards Malbec,2021.0,Terrazas de los Andes,3.9,103.0,10.99,0.6266,0.3178,...,1,0.706715,0.745839,0.327146,0.426016,0.495763,1.000000,0.377909,0.441558,0.239883
